# unialg Explorer

A gentle tour of the current `unialg` API.

A **morphism** is a typed arrow. You can run it, compose it, branch it, add parameters, add effects, and lift it through reusable data shapes.

The notebook keeps backend setup in `explore_support.py` so the main path can stay focused on the algebraic API.

## Setup

Run this once before the examples. It loads the local package and imports a small set of demo morphisms.

In [2]:
import sys
_VENV = "/home/scanbot/unialg/.venv/lib/python3.12/site-packages"
_SRC = "/home/scanbot/unialg/src"
for _p in (_VENV, _SRC):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from unialg import (
    identity, copy, delete, fst, snd, inl, inr,
    compose, par, pair, case,
    Id, One, Const, Sum, Prod,
    lower,
)
from unialg.functors import apply_poly
from unialg.actions import poly_fmap
from explore_support import *

print("Ready.")

Ready.


## 1. Plain Morphisms

`add1` and `double` are ordinary morphisms from `Int` to `Int`. The helper `show_morphism` prints the logical type, and `run_int` runs a morphism on an integer input.

In [3]:
print("add1: ", show_morphism(add1))
print("double:", show_morphism(double))

print("add1(5)  =", run_int(add1, 5))
print("double(5) =", run_int(double, 5))

add1:  Int -> Int
double: Int -> Int
add1(5)  = 6
double(5) = 10


## 2. Composition

`compose(f, g)` means: run `f`, then run `g`. The output type of `f` must match the input type of `g`.

In [4]:
add_then_double = compose(add1, double)

print("add_then_double:", show_morphism(add_then_double))
print("add_then_double(3) =", run_int(add_then_double, 3))

add_then_double: Int -> Int
add_then_double(3) = 8


In [5]:
try:
    compose(add1, add_pair)
except Exception as e:
    print(type(e).__name__ + ": add1 returns Int, but add_pair expects Int x Int")

MorphismError: add1 returns Int, but add_pair expects Int x Int


## 3. Products

Products let us talk about pairs.

`copy` duplicates one input. `fst` and `snd` project from a pair. `pair(f, g)` sends the same input to both morphisms. `par(f, g)` sends the left input to `f` and the right input to `g`.

In [6]:
copy_int = copy(INT)
fst_int = fst(INT_PAIR)
snd_int = snd(INT_PAIR)

print("copy:", show_morphism(copy_int), "   copy(7) =", run_pair(copy_int, 7))
print("fst: ", show_morphism(fst_int), "   fst(3, 9) =", run_int_from_pair(fst_int, (3, 9)))
print("snd: ", show_morphism(snd_int), "   snd(3, 9) =", run_int_from_pair(snd_int, (3, 9)))

copy: Int -> Int x Int    copy(7) = (7, 7)
fst:  Int x Int -> Int    fst(3, 9) = 3
snd:  Int x Int -> Int    snd(3, 9) = 9


In [7]:
same_input = pair(add1, double)
side_by_side = par(add1, double)

print("pair(add1, double):", show_morphism(same_input))
print("pair(add1, double)(5) =", run_pair(same_input, 5))

print("par(add1, double): ", show_morphism(side_by_side))
print("par(add1, double)(5, 6) =", run_pair(side_by_side, (5, 6)))

pair(add1, double): Int -> Int x Int
pair(add1, double)(5) = (6, 10)
par(add1, double):  Int x Int -> Int x Int
par(add1, double)(5, 6) = (6, 12)


## 4. Sums

Sums represent a choice: a value arrives from the left side or the right side.

`case(f, g)` branches on that choice. Left values go to `f`; right values go to `g`.

In [8]:
branch = case(add1, double)

print("branch:", show_morphism(branch))
print("branch(Left 5)  =", run_left_int(branch, 5))
print("branch(Right 5) =", run_right_int(branch, 5))

branch: Int + Int -> Int
branch(Left 5)  = 6
branch(Right 5) = 10


## 5. Parameters

A parametric morphism has an extra environment value. In this example, the parameter is an integer that gets added to the input.

The important surface idea is simple: provide a parameter and an input. The support helper owns the runtime packing.

In [9]:
print("add_param:", show_morphism(add_param))
print("add_param(param=10, value=3) =", run_para_int(add_param, param=10, value=3))

add_param: param Int, input Int -> Int
add_param(param=10, value=3) = 13


In [10]:
two_params = compose(add_param, add_param_again)

print("compose(add_param, add_param_again):", show_morphism(two_params))
print("with f_param=10, g_param=20, value=3:",
      run_composed_para_int(two_params, f_param=10, g_param=20, value=3))

compose(add_param, add_param_again): param Int x Int, input Int -> Int
with f_param=10, g_param=20, value=3: 33


## 6. Effects

A lax morphism returns its result inside an effect.

Here `Maybe` means the result is wrapped as an optional value, and `List` means the morphism can return multiple results. Composition sequences those effects automatically.

In [11]:
maybe_pipeline = compose(maybe_add1, maybe_double)

print("maybe_pipeline:", show_morphism(maybe_pipeline))
print("maybe_pipeline(3) =", run_maybe_int(maybe_pipeline, 3))

maybe_pipeline: Int -> Maybe Int
maybe_pipeline(3) = 8


In [12]:
list_pipeline = compose(list_step, list_double)

print("list_pipeline:", show_morphism(list_pipeline))
print("list_pipeline(3) =", run_list_int(list_pipeline, 3))

list_pipeline: Int -> List Int
list_pipeline(3) = [6, 8]


## 7. Shapes / Polynomial Functors

A polynomial functor is a reusable data shape. `Id()` marks the places where values of the current type live.

`poly_fmap(shape, h)` lifts a morphism through the shape, applying `h` at every `Id()` position.

In [13]:
pair_shape = Prod(Id(), Id())

print("shape:", pair_shape.pretty())
print("shape applied to Int:", show_type(apply_poly(pair_shape, INT)))

shape: X * X
shape applied to Int: Int x Int


In [14]:
lifted = poly_fmap(pair_shape, add1)

print("lifted add1:", show_morphism(lifted))
print("lifted add1 on (3, 7) =", run_pair(lifted, (3, 7)))

lifted add1: Int x Int -> Int x Int
lifted add1 on (3, 7) = (4, 8)


In [15]:
lifted_param = poly_fmap(pair_shape, add_param)

print("lifted add_param:", show_morphism(lifted_param))
print("param=10, value=(3, 7):", run_para_pair(lifted_param, param=10, value=(3, 7)))

lifted add_param: param Int, input Int x Int -> Int x Int
param=10, value=(3, 7): (13, 17)


In [16]:
lifted_maybe = poly_fmap(pair_shape, maybe_add1)

print("lifted maybe_add1:", show_morphism(lifted_maybe))
print("lifted maybe_add1 on (3, 7) =", run_maybe_pair(lifted_maybe, (3, 7)))

lifted maybe_add1: Int x Int -> Maybe (Int x Int)
lifted maybe_add1 on (3, 7) = (4, 8)


In [17]:
sum_shape = Sum(Id(), Id())
lifted_sum = poly_fmap(sum_shape, add1)

print("sum shape:", sum_shape.pretty())
print("Left 5  ->", run_sum_int(lifted_sum, side="left", value=5))
print("Right 5 ->", run_sum_int(lifted_sum, side="right", value=5))

sum shape: X + X
Left 5  -> Left 6
Right 5 -> Right 6


## 8. Lowering

The examples above use friendly runners. Underneath, `lower(morphism)` compiles the morphism expression into a raw Hydra term. `run(morphism, argument, ctx, graph)` applies that term and reduces it.

You usually do not need to look at the lowered term while using the API, but it is useful when checking the compiler boundary.

In [18]:
compiled = lower(add_then_double)

print("expression:", add_then_double.node.pretty())
print("lowered term class:", type(compiled).__name__)

expression: (prim ; prim)
lowered term class: TermLambda


## 9. What Can Be Composed?

`compose(f, g)` accepts `Morphism` objects. It does not accept raw Python functions directly.

A Python function can still participate, but it first has to become a Hydra `Primitive`, then a `Morphism` can point at that primitive. That backend pattern belongs in fixture/support code, not in the main tutorial path.

In [19]:
print("compose takes Morphism values:")
print("  add1 is", type(add1).__name__)
print("  lambda x: x + 1 is", type(lambda x: x + 1).__name__)

compose takes Morphism values:
  add1 is Morphism
  lambda x: x + 1 is function


## 10. Appendix: Where Did The Fixtures Come From?

The sample morphisms and notebook-friendly runners live in `explore_support.py`.

That file contains the raw Hydra details: primitive names, `expr.Prim(...)` leaves, argument packing, result unwrapping, and the `ctx` / `graph` used by `run`. Keeping that code out of the main path makes this notebook about the `unialg` API rather than Hydra plumbing.

## 11. Try It Yourself

Some small experiments:

- Compose `double` then `add3` and run it on `5`.
- Build `pair(add1, add3)` and run it on `10`.
- Build `par(double, add3)` and run it on `(2, 8)`.
- Lift `double` through `pair_shape` with `poly_fmap`.
- Compose a plain morphism with a Maybe morphism and check the result.

In [20]:
double_then_add3 = compose(double, add3)

print("double_then_add3:", show_morphism(double_then_add3))
print("double_then_add3(5) =", run_int(double_then_add3, 5))

double_then_add3: Int -> Int
double_then_add3(5) = 13


In [24]:
add3_then_double = compose(add3, double)
thendoubleagain = pair(add3_then_double, double)
print(run_int(thendoubleagain, 21))

AttributeError: 'tuple' object has no attribute 'value'